# Grounding DINO
## От текстового запроса к предразметке

Shilong Liu et al. · IDEA Research и соавторы · препринт 2023, ECCV 2024. [Публикация](https://arxiv.org/abs/2303.05499).

## Категории задаются на входе

![Текстовый запрос и оценки соответствия каждой рамки словам](assets/grounding-dino/model_explan2.PNG)

Иллюстрация авторов из [официального репозитория](https://github.com/IDEA-Research/GroundingDINO#-explanationstips-for-grounding-dino-inputs-and-outputs).

## Текст направляет поиск на трёх этапах

![Архитектура Grounding DINO: два входа, обмен признаками, выбор кандидатов и декодер](assets/grounding-dino/frameworkv4.1.png)


Liu et al., [рис. 3](https://arxiv.org/html/2303.05499v5#S2.F3). 

In [ ]:
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

from seminar.data import read_json
from seminar.models import load_detector
from seminar.metrics import nms_indices

images = read_json("data/dataset.json")
IMAGE_INDEX = 0
image = Image.open(images[IMAGE_INDEX]["image_path"]).convert("RGB")
display(image.resize((960, round(image.height * 960 / image.width))))

## Запрос и первый прогон
Названия пишем по-английски и разделяем точками. Это перечень того, что ищем. 

In [ ]:
processor, model = load_detector(device="cpu")

def predict(prompt):
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model(**inputs)
    return inputs, outputs

PROMPT = "car. truck. bus. bicycle. motorcycle."
started = time.perf_counter()
inputs, outputs = predict(PROMPT)
print(f"Прогон модели: {time.perf_counter() - started:.1f} с")
print("Рамки:", tuple(outputs.pred_boxes.shape))
print("Оценки по токенам:", tuple(outputs.logits.shape))

In [ ]:
token_scores = outputs.logits[0].sigmoid()
box_scores = token_scores.max(dim=-1).values
best = box_scores.argmax().item()
token_ids = inputs.input_ids[0].tolist()
pd.DataFrame({
    "token": processor.tokenizer.convert_ids_to_tokens(token_ids),
    "score": token_scores[best, :len(token_ids)].cpu().tolist(),
}).round(3)

## Координаты, подписи и scores
Постобработка оставляет рамки по `threshold`, собирает подписи по `text_threshold` и переводит координаты в пиксели `(x1, y1, x2, y2)`. В `target_sizes` порядок — **высота, ширина**.

In [ ]:
def postprocess(inputs, outputs, threshold=0.25, text_threshold=0.25):
    return processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=threshold, text_threshold=text_threshold,
        target_sizes=[(image.height, image.width)],
    )[0]

result = postprocess(inputs, outputs)
table = pd.DataFrame(result["boxes"].cpu().tolist(), columns=["x1", "y1", "x2", "y2"])
table["phrase"] = result["text_labels"]
table["score"] = result["scores"].cpu().tolist()
table.round(3)

In [ ]:
def show_boxes(ax, result, title):
    ax.imshow(image)
    for box, score, label in zip(result["boxes"].cpu().tolist(),
                                 result["scores"].cpu().tolist(), result["text_labels"]):
        x1, y1, x2, y2 = box
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                              fill=False, edgecolor="orange", linewidth=2))
        ax.text(x1, y1, f"{label} {score:.2f}", fontsize=8, color="black",
                bbox={"facecolor": "orange", "alpha": 0.8, "pad": 1})
    ax.set_title(f"{title} · {len(result['boxes'])} рамок")
    ax.axis("off")

fig, ax = plt.subplots(figsize=(14, 7))
show_boxes(ax, result, PROMPT)
plt.show()

## Порог рамок: полнота и лишние кандидаты
Сравним три порога на **одном прогоне**. При снижении порога появляются дополнительные кандидаты: среди них могут быть и пропуски, и ложные срабатывания. 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, threshold in zip(axes, [0.15, 0.25, 0.40]):
    show_boxes(ax, postprocess(inputs, outputs, threshold=threshold), f"threshold={threshold}")
plt.tight_layout()
plt.show()

## Порог текста: состав подписи

In [ ]:
rows = []
for text_threshold in [0.15, 0.25, 0.45]:
    r = postprocess(inputs, outputs, text_threshold=text_threshold)
    rows.append({"text_threshold": text_threshold, "boxes": len(r["boxes"]),
                 "phrases": r["text_labels"]})
pd.DataFrame(rows)

## Запрос меняет выдачу

In [ ]:
prompts = ["vehicle.", "car.", PROMPT]
fig, axes = plt.subplots(1, len(prompts), figsize=(21, 7))
for ax, prompt in zip(axes, prompts):
    prompt_inputs, prompt_outputs = predict(prompt)
    show_boxes(ax, postprocess(prompt_inputs, prompt_outputs), prompt)
plt.tight_layout()
plt.show()

## NMS: фильтрация дубликатов
Постобработка выше не выполняет NMS. Если один объект получил несколько почти одинаковых рамок, оставим рамку с большим score. NMS сравнивает IoU — площадь пересечения, делённую на площадь объединения. Слишком низкий порог NMS может удалить разные объекты, которые перекрывают друг друга.

In [ ]:
candidates = postprocess(inputs, outputs, threshold=0.15)
keep = nms_indices(candidates["boxes"].cpu().tolist(),
                   candidates["scores"].cpu().tolist(), threshold=0.8)
filtered = {"boxes": candidates["boxes"][keep], "scores": candidates["scores"][keep],
            "text_labels": [candidates["text_labels"][i] for i in keep]}
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
show_boxes(axes[0], candidates, "До NMS")
show_boxes(axes[1], filtered, "После NMS, IoU=0.8")
plt.tight_layout()
plt.show()

In [ ]:
from seminar.api import boxes_to_annotations

normalized = filtered["boxes"].cpu() / torch.tensor([image.width, image.height, image.width, image.height])
normalized = normalized.clamp(0, 1).tolist()
boxes_to_annotations(normalized)[:2]